<a href="https://colab.research.google.com/github/AngeloSorte/AI-Ethics-Checker/blob/angelosorte.github.io/AI_Ethics_Checker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install necessary libraries
# transformers: for NLP models
# torch: backend for the models
# pandas: for handling data

!pip install transformers torch pandas --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 71.8 MB/s eta 0:00:00


In [2]:
# Import the libraries

import torch
from transformers import pipeline
import pandas as pd

In [3]:
# Create a text classification pipeline
# We use a pre-trained model for detecting toxic content
# This model will help us flag potentially unethical text

ethics_checker = pipeline(
    "text-classification",
    model="unitary/toxic-bert",
    return_all_scores=True  # we want all possible labels
)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/811 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cpu
/usr/local/lib/python3.11/dist-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [4]:
# Function to check ethical issues in a text

def check_ethics(text):
    """
    Analyze text for ethical issues using the ethics_checker model
    Returns a simple report with warnings for potential problems
    """
    results = ethics_checker(text)[0]  # get first result (the text itself)

    # Create an empty report
    report = []

    # Go through each label and check score
    for item in results:
        label = item['label']
        score = item['score']

        # Consider any score above 0.5 as a warning
        if score > 0.5:
            report.append(f"⚠️ {label} detected with confidence {score:.2f}")

    # If no warnings, text is considered "safe"
    if not report:
        report.append("✅ Text seems ethically safe.")

    return report


In [8]:
# Test the ethics checker with sample text

sample_text = "I hate this group of people and want them to disappear."
report = check_ethics(sample_text)

# Print the report
for line in report:
    print(line)


⚠️ toxic detected with confidence 0.52


In [10]:
# Interactive text input and save results
# We will ask the user to enter text, check it, and save the results

import json

def analyze_and_save():
    """
    Function to interactively check text ethics and save results to CSV and JSON
    """
    # Ask user for text input
    text = input("Enter text to analyze for ethical issues: ")

    # Analyze text
    report = check_ethics(text)

    # Show report
    print("\nEthics Report:")
    for line in report:
        print(line)

    # Save results to CSV
    df = pd.DataFrame({"text": [text], "report": [", ".join(report)]})
    df.to_csv("ethics_report.csv", index=False)

    # Save results to JSON
    with open("ethics_report.json", "w") as f:
        json.dump({"text": text, "report": report}, f, indent=4)

    print("\n✅ Results saved to ethics_report.csv and ethics_report.json")

# Run the interactive function
analyze_and_save()


Enter text to analyze for ethical issues: Love you

Ethics Report:
✅ Text seems ethically safe.

✅ Results saved to ethics_report.csv and ethics_report.json
